In [3]:
# Dalton Pedersen
# P4 / HW6
# 10/03/26

In [1]:
import os
import astropy.io.fits as fits
import numpy as np

In [2]:
datadir = 'hw6_data/'

fext = '.fits'

In [3]:
objfile = []
darkfile = []

In [4]:
for fname in os.listdir(datadir):
    #check if it is a fits file
    if fname.endswith(fext):
        #remove fits extension to get root name
        root_name = fname[:-len(fext)]

        #check if it is target or dark
        if fname.startswith('rdpharocor_stars_13s'):
            objfile.append(root_name)
        elif fname.startswith('rdpharocor_dark_13s'):
            darkfile.append(root_name)


# sort files to put in order
objfile.sort()
darkfile.sort()

In [5]:
print(f'Data Directory: {datadir}')
print(f'Fits Extension: {fext}')
print(f'Last object file: {objfile[-1] if objfile else 'None'}') # error handling for empty objfile list
print(f'Last dark file: {darkfile[-1] if darkfile else 'None'}')

Data Directory: hw6_data/
Fits Extension: .fits
Last object file: rdpharocor_stars_13s_9
Last dark file: rdpharocor_dark_13s_2


In [6]:
#make path to sample file
sample_path = os.path.join(datadir, objfile[0] + fext)

In [7]:
with fits.open(sample_path) as hdul:
    sample_data = hdul[0].data
    ny, nx = sample_data.shape

In [8]:
#this should not be hardcoded because it is very simple to determine this directly
# this allows the code to be more adaptable
nobj = len(objfile)
ndark = len(darkfile)

In [9]:
print(f'Image Dimensions (ny, nx): ({ny}, {nx})')
print(f'Number of object files: {nobj}')
print(f'Number of dark files: {ndark}\n')

Image Dimensions (ny, nx): (1024, 1024)
Number of object files: 9
Number of dark files: 2



In [10]:
objcube = np.zeros((nobj, ny, nx), dtype=np.float64)
darkcube = np.zeros((ndark, ny, nx), dtype=np.float64)

In [11]:
print(f'Object data cube shape: {objcube.shape}')
print(f'Dark data cube shape: {darkcube.shape}')

Object data cube shape: (9, 1024, 1024)
Dark data cube shape: (2, 1024, 1024)


In [12]:
objhead = None

for i, filename in enumerate(objfile):
    filepath = os.path.join(datadir, filename + fext)
    with fits.open(filepath) as hdul:
        objcube[i] = hdul[0].data.astype(np.float64)
        objhead = hdul[0].header #save last header

In [13]:
darkhead = None

for i, filename in enumerate(darkfile):
    filepath = os.path.join(datadir, filename + fext)
    with fits.open(filepath) as hdul:
        darkcube[i] = hdul[0].data.astype(np.float64)
        darkhead = hdul[0].header #save last header

In [14]:
obj_date_obs = objhead.get("DATE-OBS", "N/A")
dark_date_obs = darkhead.get("DATE-OBS", "N/A")

In [15]:
print(f'Object Image Last DATE-OBS: {obj_date_obs}')
print(f'Dark Image Last DATE-OBS: {dark_date_obs}')

Object Image Last DATE-OBS: 2002-01-27
Dark Image Last DATE-OBS: 2002-01-27


In [16]:
# Extra Credit Problem 3:
#Typically in the fits standard the date-obs is formatted as a utc timestamp.
# This would contain the date and time and looking for time-obs is redundant.

In [24]:
# HW6 Start

In [25]:
# set axis=0 to operate down the stack of dark frames without loops
master_dark = np.median(darkcube, axis=0)

In [26]:
target_pixel_val = master_dark[217, 184]
print(f'Median dark pixel [217, 184] value: {target_pixel_val}')

Median dark pixel [217, 184] value: 19.0


In [34]:
darkhead.add_history("Median-combined dark frame")
#print(darkhead["HISTORY"])

rdpharo: Wed Sep 11 09:34:23 EDT 2024 theodora_karalidi
Unpacked 512x512x4 array into 1024x1024 array
Median-combined dark frame
Median-combined dark frame
Median-combined dark frame
Median-combined dark frame


In [31]:
primary_hdu = fits.PrimaryHDU(data=master_dark, header=darkhead)
primary_hdu.writeto("dark_13s_med.fits", overwrite=True)

In [32]:
# save val before subtraction

val_before = objcube[0, 217, 184]
print(f'Object 0 pixel [217, 184] before dark subtraction: {val_before}')

objcube -= master_dark

val_after = objcube[0, 217, 184]
print(f'Object 0 pixel [217, 184] after dark subtraction: {val_after}')

Object 0 pixel [217, 184] before dark subtraction: 10841.0
Object 0 pixel [217, 184] after dark subtraction: 10822.0


In [33]:
# write first frame to a fits file

obj0_hdu = fits.PrimaryHDU(data=objcube[0])
obj0_hdu.writeto('hw6_dpedersen_prob2_graph1.fits', overwrite=True)